In [ ]:
#TODO i made some change in urnning but forgot what ai changed, likely the split fraction
#TODO need to rerun this cause of this. Affected tu berlin(knn mxied faiss and itf)
#poitentially coil 100 as well

In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from its.search import InverseTransformationSearch
from search.parallel_gradient import ParallelGradientDescent
from utils.affine_transforms_old import AffineTransformation2D
from utils.sampling import BatchNegativeSampler

#torch.cuda.is_available = lambda: False
#device = torch.device("cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]

budget = None

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]

from utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)
from experiment_thesis.main import train_and_get_model, train_or_load_energy_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "unsupervised_metrics")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
#chek if data is iamge data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
from utils.transforms.apply import grid_resample
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="sobol"
    ).to(device)

In [ ]:
from experiment_thesis.dataset_preperation.basic_networks import make_deterministic
make_deterministic(model)



In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_layer_embedding_cache_config,create_layer_embedding_cache
cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import json
import torch
import optuna
import gc

from experiment_thesis.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO


# ---------------- CONFIG ----------------
detectors = ["knn", "knn_mixed","knn_mixed_faiss","knn_itf","vim"]
search_objective = "search"

optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
optimizer_search_small = SHGO(initial_samples=10, local_runs=1, local_max_steps=0)
optimizer_search_large = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)



In [ ]:
if dataset in ["mnist", "bigger_mnist", "emnist", "bigger_emnist"]:
        report_fraction_stage1 = 0.2
else:
        report_fraction_stage1 = 0.4

In [ ]:
import os
import json
import torch
import gc
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
import optuna

from experiment_thesis.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO

search_objective = "search"

n_trials_search = 100
n_trials_search_refine = 10
eval_repeats = 8
show_progress = True
top_k = 6
transform_seq_arg = transform_seq

optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "ood_studies_v1_one_loader",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(base_results_dir, exist_ok=True)

model.eval().to(device)

# ---------------- Prepare validation subset ----------------
val_dataset = val_loader_transformed.dataset
n_samples = len(val_dataset)
subset_size = n_samples // 6

rng = np.random.default_rng(seed=42)
all_indices = rng.permutation(n_samples)

val_subset_1 = Subset(val_dataset, all_indices[:subset_size])

val_loader_small_1 = DataLoader(
    val_subset_1,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

val_loader_transformed_preshuffled = DataLoader(
    val_dataset,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

print(f"Each subset: {subset_size} samples ({subset_size / n_samples:.1%} of dataset)")
print("One disjoint subset created.\n")

# ---------------- Loop over detectors ----------------
for detector in detectors:
    print(f"\n=== Detector (V1): {detector} ===")
    detector_dir = os.path.join(base_results_dir, detector)
    os.makedirs(detector_dir, exist_ok=True)

    params_path = os.path.join(detector_dir, "best_params.json")
    eval_path = os.path.join(detector_dir, "eval_results.json")

    # ---------------- Load or run search ----------------
    if os.path.exists(params_path):
        print(f"[{detector}] Found existing best_params.json, skipping search.")
        try:
            with open(params_path, "r") as f:
                best_params = json.load(f)
        except Exception as e:
            print(f"[{detector}] Warning: Failed to read best_params.json ({e}), using default params.")
            best_params = get_default_ood_params(detector)
    else:
        gc.collect(); torch.cuda.empty_cache()
        default_params = get_default_ood_params(detector)
        best_stage1_params_all = []

        val_loaders_small = [val_loader_small_1]
        trials_per_loader = [n_trials_search]

        # === Stage 1: coarse search ===
        for i, (small_loader, n_trials_this_run) in enumerate(zip(val_loaders_small, trials_per_loader), start=1):
            print(f"\n[{detector}] Running on coarse loader {i}/1 ({n_trials_this_run} trials)...")

            optimizer = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
            objective_kwargs_stage1 = {
                "optimizer": optimizer,
                "model": model,
                "train_cache": train_cache,
                "val_loader": small_loader,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "report_fraction": report_fraction_stage1,
                "repeats": 1,
            }

            study_stage1 = run_ood_study(
                study_name=f"{detector}_v1_stage1_part{i}",
                storage_path=None,
                detector_name=detector,
                objective_type=search_objective,
                objective_kwargs=objective_kwargs_stage1,
                n_trials=n_trials_this_run,
                enqueue_params=[copy.deepcopy(default_params)],
            )

            if study_stage1 is not None:
                completed_trials = [t for t in study_stage1.trials if t.state == optuna.trial.TrialState.COMPLETE]
                if not completed_trials:
                    print(f"[{detector}] Warning: No completed trials in subset {i}")
                    continue

                topk_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:top_k]
                for t in topk_trials:
                    best_stage1_params_all.append(copy.deepcopy(t.params))

        # === Stage 2: refine search ===
        print(f"\n[{detector}] Stage 2: refine search...")
        optimizer_stage2 = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
        objective_kwargs_stage2 = {
            "optimizer": optimizer_stage2,
            "model": model,
            "train_cache": train_cache,
            "val_loader": val_loader_transformed_preshuffled,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "report_fraction": 0.1,
            "repeats": 1,
        }

        enqueue_list = [copy.deepcopy(default_params)] + [copy.deepcopy(p) for p in best_stage1_params_all]
        study_stage2 = run_ood_study(
            study_name=f"{detector}_v1_stage2",
            storage_path=None,
            detector_name=detector,
            objective_type=search_objective,
            objective_kwargs=objective_kwargs_stage2,
            n_trials=n_trials_search_refine,
            enqueue_params=enqueue_list,
        )

        best_params = (
            get_best_ood_params_from_study(study_stage2)
            if study_stage2
            else (best_stage1_params_all[-1] if best_stage1_params_all else default_params)
        )

        with open(params_path, "w") as f:
            json.dump(best_params, f, indent=2)
        print(f"[{detector}] Saved best parameters to {params_path}")

    # ---------------- Evaluate ----------------
    run_evaluation = True
    if os.path.exists(eval_path):
        try:
            with open(eval_path, "r") as f:
                eval_data = json.load(f)
            if eval_data.get("number_of_runs", 0) >= eval_repeats:
                print(f"[{detector}] Evaluation already complete, skipping.")
                run_evaluation = False
        except Exception as e:
            print(f"[{detector}] Warning: Could not read eval JSON ({e}), re-running evaluation.")

    if run_evaluation:
        print(f"[{detector}] Evaluating final configuration...")
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
        )

        print(
            f"[{detector}] Final Search Accuracy (V1): "
            f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
        )

print("\nAll detectors processed successfully (V1).")


In [ ]:
import os
import json
import torch
import gc
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
import optuna

from experiment_thesis.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO

search_objective = "search"

n_trials_search_v2 = 100
n_trials_search_refine = 10
eval_repeats = 8
show_progress = True
top_k = 3
transform_seq_arg = transform_seq

optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "ood_studies_v2_two_loaders",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(base_results_dir, exist_ok=True)

model.eval().to(device)

# ---------------- Prepare validation subsets ----------------
val_dataset = val_loader_transformed.dataset
n_samples = len(val_dataset)
subset_size = n_samples // 6

rng = np.random.default_rng(seed=42)
all_indices = rng.permutation(n_samples)

val_subsets = [
    Subset(val_dataset, all_indices[i*subset_size:(i+1)*subset_size])
    for i in range(2)
]

val_loaders_small = [
    DataLoader(
        subset,
        batch_size=dataset_info.batch_size,
        shuffle=False,
        num_workers=val_loader_transformed.num_workers,
        pin_memory=True,
        persistent_workers=val_loader_transformed.persistent_workers
    )
    for subset in val_subsets
]

val_loader_transformed_preshuffled = DataLoader(
    val_dataset,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

print(f"Each subset: {subset_size} samples ({subset_size / n_samples:.1%} of dataset)")
print("Two disjoint subsets created.\n")

# ---------------- Loop over detectors ----------------
for detector in detectors:
    print(f"\n=== Detector (V2): {detector} ===")
    detector_dir = os.path.join(base_results_dir, detector)
    os.makedirs(detector_dir, exist_ok=True)

    params_path = os.path.join(detector_dir, "best_params.json")
    eval_path = os.path.join(detector_dir, "eval_results.json")

    # ---------------- Load or run search ----------------
    if os.path.exists(params_path):
        print(f"[{detector}] Found existing best_params.json, skipping search.")
        try:
            with open(params_path, "r") as f:
                best_params = json.load(f)
        except Exception as e:
            print(f"[{detector}] Warning: Failed to read best_params.json ({e}), using default params.")
            best_params = get_default_ood_params(detector)
    else:
        gc.collect(); torch.cuda.empty_cache()
        default_params = get_default_ood_params(detector)
        best_stage1_params_all = []

        trials_per_loader = [n_trials_search_v2 // 2, n_trials_search_v2 - n_trials_search_v2 // 2]

        # === Stage 1: coarse search on subsets ===
        for i, (small_loader, n_trials_this_run) in enumerate(zip(val_loaders_small, trials_per_loader), start=1):
            print(f"\n[{detector}] Running on coarse loader {i}/2 ({n_trials_this_run} trials)...")

            optimizer = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
            objective_kwargs_stage1 = {
                "optimizer": optimizer,
                "model": model,
                "train_cache": train_cache,
                "val_loader": small_loader,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "report_fraction": report_fraction_stage1,
                "repeats": 1,
            }

            study_stage1 = run_ood_study(
                study_name=f"{detector}_v2_stage1_part{i}",
                storage_path=None,
                detector_name=detector,
                objective_type=search_objective,
                objective_kwargs=objective_kwargs_stage1,
                n_trials=n_trials_this_run,
                enqueue_params=[copy.deepcopy(default_params)],
            )

            if study_stage1 is not None:
                completed_trials = [t for t in study_stage1.trials if t.state == optuna.trial.TrialState.COMPLETE]
                if not completed_trials:
                    print(f"[{detector}] Warning: No completed trials in subset {i}")
                    continue

                topk_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:top_k]
                for t in topk_trials:
                    best_stage1_params_all.append(copy.deepcopy(t.params))

        # === Stage 2: refine search ===
        print(f"\n[{detector}] Stage 2: refine search...")
        optimizer_stage2 = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
        objective_kwargs_stage2 = {
            "optimizer": optimizer_stage2,
            "model": model,
            "train_cache": train_cache,
            "val_loader": val_loader_transformed_preshuffled,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "report_fraction": 0.1,
            "repeats": 1,
        }

        enqueue_list = [copy.deepcopy(default_params)] + [copy.deepcopy(p) for p in best_stage1_params_all]
        study_stage2 = run_ood_study(
            study_name=f"{detector}_v2_stage2",
            storage_path=None,
            detector_name=detector,
            objective_type=search_objective,
            objective_kwargs=objective_kwargs_stage2,
            n_trials=n_trials_search_refine,
            enqueue_params=enqueue_list,
        )

        best_params = (
            get_best_ood_params_from_study(study_stage2)
            if study_stage2
            else (best_stage1_params_all[-1] if best_stage1_params_all else default_params)
        )

        with open(params_path, "w") as f:
            json.dump(best_params, f, indent=2)
        print(f"[{detector}] Saved best parameters to {params_path}")

    # ---------------- Evaluate ----------------
    run_evaluation = True
    if os.path.exists(eval_path):
        try:
            with open(eval_path, "r") as f:
                eval_data = json.load(f)
            if eval_data.get("number_of_runs", 0) >= eval_repeats:
                print(f"[{detector}] Evaluation already complete, skipping.")
                run_evaluation = False
        except Exception as e:
            print(f"[{detector}] Warning: Could not read eval JSON ({e}), re-running evaluation.")

    if run_evaluation:
        print(f"[{detector}] Evaluating final configuration...")
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
        )

        print(
            f"[{detector}] Final Search Accuracy (V2): "
            f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
        )

print("\nAll detectors processed successfully (V2).")


In [ ]:
import os
import json
import torch
import gc
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
import optuna

from experiment_thesis.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO

search_objective = "search"

n_trials_search_v3 = 100
n_trials_search_refine = 10
eval_repeats = 8
show_progress = True
top_k = 2
transform_seq_arg = transform_seq

optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "ood_studies_v3_three_loaders",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(base_results_dir, exist_ok=True)

model.eval().to(device)

# ---------------- Create 3 disjoint 1/6 subsets ----------------
val_dataset = val_loader_transformed.dataset
n_samples = len(val_dataset)
subset_size = n_samples // 6

rng = np.random.default_rng(seed=42)
all_indices = rng.permutation(n_samples)

subsets = [
    Subset(val_dataset, all_indices[i*subset_size:(i+1)*subset_size])
    for i in range(3)
]

val_loaders_small = [
    DataLoader(
        subset,
        batch_size=dataset_info.batch_size,
        shuffle=False,
        num_workers=val_loader_transformed.num_workers,
        pin_memory=True,
        persistent_workers=val_loader_transformed.persistent_workers,
    )
    for subset in subsets
]

val_loader_transformed_preshuffled = DataLoader(
    val_dataset,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers,
)

print(f"Each subset: {subset_size} samples ({subset_size / n_samples:.1%} of dataset)")
print("Three disjoint subsets created.\n")

# ---------------- Loop over detectors ----------------
for detector in detectors:
    print(f"\n=== Detector (V3): {detector} ===")
    detector_dir = os.path.join(base_results_dir, detector)
    os.makedirs(detector_dir, exist_ok=True)
    params_path = os.path.join(detector_dir, "best_params.json")
    eval_path = os.path.join(detector_dir, "eval_results.json")

    # ---------------- Load or run search ----------------
    if os.path.exists(params_path):
        print(f"[{detector}] Found existing best_params.json, skipping search.")
        try:
            with open(params_path, "r") as f:
                best_params = json.load(f)
        except Exception as e:
            print(f"[{detector}] Warning: Failed to read best_params.json ({e}), using default params.")
            best_params = get_default_ood_params(detector)
    else:
        # === Stage 1: coarse search on subsets ===
        gc.collect(); torch.cuda.empty_cache()
        default_params = get_default_ood_params(detector)
        best_stage1_params_all = []

        trials_per_loader = [n_trials_search_v3 // 3] * 3
        trials_per_loader[-1] += n_trials_search_v3 - sum(trials_per_loader)

        for i, (small_loader, n_trials_this_run) in enumerate(zip(val_loaders_small, trials_per_loader), start=1):
            print(f"\n[{detector}] Coarse loader {i}/3 ({n_trials_this_run} trials)...")

            optimizer = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
            objective_kwargs_stage1 = {
                "optimizer": optimizer,
                "model": model,
                "train_cache": train_cache,
                "val_loader": small_loader,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "report_fraction": report_fraction_stage1,
                "repeats": 1,
            }

            study_stage1 = run_ood_study(
                study_name=f"{detector}_v3_stage1_part{i}",
                storage_path=None,
                detector_name=detector,
                objective_type=search_objective,
                objective_kwargs=objective_kwargs_stage1,
                n_trials=n_trials_this_run,
                enqueue_params=[copy.deepcopy(default_params)],
            )

            if study_stage1 is not None:
                completed_trials = [t for t in study_stage1.trials if t.state == optuna.trial.TrialState.COMPLETE]
                if not completed_trials:
                    print(f"[{detector}] Warning: No completed trials in subset {i}")
                    continue

                topk_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:top_k]
                for t in topk_trials:
                    best_stage1_params_all.append(copy.deepcopy(t.params))

        # === Stage 2: refine search ===
        print(f"\n[{detector}] Stage 2: refine search...")
        optimizer_stage2 = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)
        objective_kwargs_stage2 = {
            "optimizer": optimizer_stage2,
            "model": model,
            "train_cache": train_cache,
            "val_loader": val_loader_transformed_preshuffled,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "report_fraction": 0.1,
            "repeats": 1,
        }

        enqueue_list = [copy.deepcopy(default_params)] + [copy.deepcopy(p) for p in best_stage1_params_all]
        study_stage2 = run_ood_study(
            study_name=f"{detector}_v3_stage2",
            storage_path=None,
            detector_name=detector,
            objective_type=search_objective,
            objective_kwargs=objective_kwargs_stage2,
            n_trials=n_trials_search_refine,
            enqueue_params=enqueue_list,
        )

        best_params = (
            get_best_ood_params_from_study(study_stage2)
            if study_stage2
            else (best_stage1_params_all[-1] if best_stage1_params_all else default_params)
        )

        with open(params_path, "w") as f:
            json.dump(best_params, f, indent=2)
        print(f"[{detector}] Saved best parameters to {params_path}")

    # ---------------- Evaluate ----------------
    run_evaluation = True
    if os.path.exists(eval_path):
        try:
            with open(eval_path, "r") as f:
                eval_data = json.load(f)
            if eval_data.get("number_of_runs", 0) >= eval_repeats:
                print(f"[{detector}] Evaluation already complete, skipping.")
                run_evaluation = False
        except Exception as e:
            print(f"[{detector}] Warning: Could not read eval JSON ({e}), re-running evaluation.")

    if run_evaluation:
        print(f"[{detector}] Evaluating best config...")
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
        )

        print(
            f"[{detector}] Final Search Accuracy (V3): "
            f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
        )

print("\nAll detectors processed successfully (V3).")


In [ ]:
import os
import json
import torch
import gc
import copy

from experiment_thesis.ood.base_prepare import (
    run_ood_study_halving,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.shgo import SHGO

search_objective = "search"
n_trials_halving = 60
eval_repeats = 8
show_progress = True
transform_seq_arg = transform_seq

optimizer_search_eval = SHGO(initial_samples=60, local_runs=1, local_max_steps=0)

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "ood_studies_halving_full_loader",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(base_results_dir, exist_ok=True)

model.eval().to(device)

for detector in detectors:
    print(f"\n=== Detector (Halving): {detector} ===")
    detector_dir = os.path.join(base_results_dir, detector)
    os.makedirs(detector_dir, exist_ok=True)
    params_path = os.path.join(detector_dir, "best_params.json")
    eval_path = os.path.join(detector_dir, "eval_results.json")

    # Load or run search only if best_params.json does NOT exist
    if os.path.exists(params_path):
        print(f"[{detector}] Found existing best_params.json, skipping search.")
        try:
            with open(params_path, "r") as f:
                best_params = json.load(f)
        except Exception as e:
            print(f"[{detector}] Warning: Failed to read best_params.json ({e}), using default params.")
            best_params = get_default_ood_params(detector)
    else:
        gc.collect(); torch.cuda.empty_cache()
        default_params = get_default_ood_params(detector)

        objective_kwargs = {
            "optimizer": optimizer_search_eval,
            "model": model,
            "train_cache": train_cache,
            "val_loader": val_loader_transformed_preshuffled,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "report_fraction": 0.1,
            "repeats": 1,
        }

        print(f"[{detector}] Running successive halving optimization ({n_trials_halving} trials)...")
        study = run_ood_study_halving(
            study_name=f"{detector}_halving_full_loader",
            storage_path=None,
            detector_name=detector,
            objective_type=search_objective,
            objective_kwargs=objective_kwargs,
            n_trials=n_trials_halving,
            enqueue_params=[copy.deepcopy(default_params)],
        )

        best_params = get_best_ood_params_from_study(study) if study else default_params

        with open(params_path, "w") as f:
            json.dump(best_params, f, indent=2)
        print(f"[{detector}] Saved best parameters to {params_path}")

    # Evaluate if eval_results.json missing or incomplete
    run_evaluation = True
    if os.path.exists(eval_path):
        try:
            with open(eval_path, "r") as f:
                eval_data = json.load(f)
            if eval_data.get("number_of_runs", 0) >= eval_repeats:
                print(f"[{detector}] Evaluation already complete, skipping.")
                run_evaluation = False
        except Exception as e:
            print(f"[{detector}] Warning: Could not read eval JSON ({e}), re-running evaluation.")

    if run_evaluation:
        print(f"[{detector}] Evaluating best config...")
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
        )

        print(
            f"[{detector}] Final Search Accuracy (Halving): "
            f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
        )

print("\nAll detectors processed successfully (Halving).")


In [ ]:
import pandas as pd

detectors = ["knn", "knn_mixed", "knn_mixed_faiss", "knn_itf","vim"]

# === LOAD RESULTS ===
def load_metrics(result_dir, detectors):
    results = {}
    for det in detectors:
        eval_path = os.path.join(result_dir, det, "eval_results.json")
        print(f"Loading {eval_path}...")
        if os.path.exists(eval_path):
            with open(eval_path, "r") as f:
                data = json.load(f)
                results[det] = {
                    "accuracy_mean": data.get("accuracy_mean", np.nan),
                    "accuracy_se": data.get("accuracy_se", np.nan),
                }

    return results

metrics_v1 = load_metrics(
    os.path.join(
        current_path,
        "experiment_files",
        "ood_studies_v1_one_loader",
        str(dataset),
        str(architecture),
        getattr(dataset_info, "transform_seq_name", "default"),
    ),
    detectors,)
metrics_v2 = load_metrics(
    os.path.join(
        current_path,
        "experiment_files",
        "ood_studies_v2_two_loaders",
        str(dataset),
        str(architecture),
        getattr(dataset_info, "transform_seq_name", "default"),
    ),
    detectors,)
metrics_v3 = load_metrics(
    os.path.join(
        current_path,
        "experiment_files",
        "ood_studies_v3_three_loaders",
        str(dataset),
        str(architecture),
        getattr(dataset_info, "transform_seq_name", "default"),
    ),
    detectors,)
metrics_halving = load_metrics(
    os.path.join(
        current_path,
        "experiment_files",
        "ood_studies_halving_full_loader",
        str(dataset),
        str(architecture),
        getattr(dataset_info, "transform_seq_name", "default"),
    ),
    detectors,)


# === CREATE COMPARISON TABLE ===
comparison_data = []
for det in detectors:
    v1 = metrics_v1[det]
    v2 = metrics_v2[det]
    v3 = metrics_v3[det]
    v4 = metrics_halving[det]
    comparison_data.append({
        "Detector": det,
        "V1_Accuracy": v1["accuracy_mean"],
        "V1_se": v1["accuracy_se"],
        "V2_Accuracy": v2["accuracy_mean"],
        "V2_se": v2["accuracy_se"],
        "V3_Accuracy": v3["accuracy_mean"],
        "V3_se": v3["accuracy_se"],
        "V4_Accuracy": v4["accuracy_mean"],
        "V4_se": v4["accuracy_se"],
        "Δ_Accuracy_V3_V1": v3["accuracy_mean"] - v1["accuracy_mean"],
        "Δ_Accuracy_V3_V2": v3["accuracy_mean"] - v2["accuracy_mean"],
        "Δ_Accuracy": v2["accuracy_mean"] - v1["accuracy_mean"],

    })

df_compare = pd.DataFrame(comparison_data)
print("\n=== OOD Detector Comparison ===")
print(df_compare.to_string(index=False))



In [ ]:
# === VISUALIZE RESULTS ===
x = np.arange(len(detectors))
width = 0.22

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - 1.5*width, df_compare["V1_Accuracy"], width, label='V1', color='skyblue', alpha=0.7)
ax.bar(x-0.5 *width, df_compare["V2_Accuracy"], width, label='V2', color='salmon', alpha=0.7)
ax.bar(x +0.5 *width, df_compare["V3_Accuracy"], width, label='V3', color='lightgreen', alpha=0.7)
#print error bars
ax.errorbar(x - 1.5*width, df_compare["V1_Accuracy"], yerr=df_compare["V1_se"], fmt='none', ecolor='blue', capsize=5)
ax.errorbar(x - 0.5*width, df_compare["V2_Accuracy"], yerr=df_compare["V2_se"], fmt='none', ecolor='red', capsize=5)
ax.errorbar(x + 0.5*width, df_compare["V3_Accuracy"], yerr=df_compare["V3_se"], fmt='none', ecolor='green', capsize=5)
ax.bar(x + 1.5*width, df_compare["V4_Accuracy"], width, label='Halving', color='orange', alpha=0.7)
ax.errorbar(x + 1.5*width, df_compare["V4_Accuracy"], yerr=df_compare["V4_se"], fmt='none', ecolor='darkorange', capsize=5)


ax.set_ylabel('Accuracy')
ax.set_title(f'OOD Detector Accuracy Comparison ({dataset}, {architecture})')
ax.set_xticks(x)
ax.set_xticklabels(detectors, rotation=25)
min_accuracy = df_compare[["V1_Accuracy","V2_Accuracy","V3_Accuracy"]].min().min()
max_accuracy = df_compare[["V1_Accuracy","V2_Accuracy","V3_Accuracy"]].max().max()
ax.set_ylim(min_accuracy - 0.05, max_accuracy + 0.05)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

#calculate the average accuracy per version
avg_v1 = df_compare["V1_Accuracy"].mean()
avg_v2 = df_compare["V2_Accuracy"].mean()
avg_v3 = df_compare["V3_Accuracy"].mean()
avg_v4 = df_compare["V4_Accuracy"].mean()
print(f"Average Accuracy V1: {avg_v1:.4f}")
print(f"Average Accuracy V2: {avg_v2:.4f}")
print(f"Average Accuracy V3: {avg_v3:.4f}")
print(f"Average Accuracy V4: {avg_v4:.4f}")

In [ ]:
# === Δ Accuracy Plot ===
plt.figure(figsize=(8, 4))
plt.bar(detectors, df_compare["Δ_Accuracy_V3_V1"], color='blue', alpha=0.7)
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel("Δ Accuracy (V3 - V1)")
plt.title("Performance Improvement per Detector")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
for dataset_vis in ["bigger_mnist","bigger_emnist","coil100","tu_berlin","modelnet10"]:
    dataset2 = dataset_vis
    architecture = default_architecutre_mapping[dataset2]

    experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
    dataset_info2 = get_dataset_info(dataset2)


    detectors = ["knn", "knn_mixed", "knn_mixed_faiss", "knn_itf","vim"]

    # === LOAD RESULTS ===
    def load_metrics(result_dir, detectors):
        results = {}
        for det in detectors:
            eval_path = os.path.join(result_dir, det, "eval_results.json")
            print(f"Loading {eval_path}...")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    data = json.load(f)
                    results[det] = {
                        "accuracy_mean": data.get("accuracy_mean", np.nan),
                        "accuracy_std": data.get("accuracy_std", np.nan),
                    }

        return results

    metrics_v1 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v1_one_loader",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors,)
    metrics_v2 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v2_two_loaders",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors,)
    metrics_v3 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v3_three_loaders",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors,)

    metrics_halving = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_halving_full_loader",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors,)


    # === CREATE COMPARISON TABLE ===
    comparison_data = []
    for det in detectors:
        v1 = metrics_v1[det]
        v2 = metrics_v2[det]
        v3 = metrics_v3[det]
        v4 = metrics_halving[det]
        comparison_data.append({
            "Detector": det,
            "V1_Accuracy": v1["accuracy_mean"],
            "V1_Std": v1["accuracy_std"],
            "V2_Accuracy": v2["accuracy_mean"],
            "V2_Std": v2["accuracy_std"],
            "V3_Accuracy": v3["accuracy_mean"],
            "V3_Std": v3["accuracy_std"],
            "V4_Accuracy": v4["accuracy_mean"],
            "V4_Std": v4["accuracy_std"],
            "Δ_Accuracy_V3_V1": v3["accuracy_mean"] - v1["accuracy_mean"],
            "Δ_Accuracy_V3_V2": v3["accuracy_mean"] - v2["accuracy_mean"],
            "Δ_Accuracy": v2["accuracy_mean"] - v1["accuracy_mean"],
        })

    df_compare = pd.DataFrame(comparison_data)
    print("\n=== OOD Detector Comparison ===")
    print(df_compare.to_string(index=False))

    # === VISUALIZE RESULTS ===
    x = np.arange(len(detectors))
    width = 0.2

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - 1.5*width, df_compare["V1_Accuracy"], width, label='V1', color='skyblue', alpha=0.7)
    ax.bar(x-0.5 *width, df_compare["V2_Accuracy"], width, label='V2', color='salmon', alpha=0.7)
    ax.bar(x +0.5 *width, df_compare["V3_Accuracy"], width, label='V3', color='lightgreen', alpha=0.7)
    ax.bar(x + 1.5*width, df_compare["V4_Accuracy"], width, label='Halving', color='orange', alpha=0.7)
    #print error bars

    ax.set_ylabel('Accuracy')
    ax.set_title(f'OOD Detector Accuracy Comparison ({dataset}, {architecture})')
    ax.set_xticks(x)
    ax.set_xticklabels(detectors, rotation=25)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # === Δ Accuracy Plot ===
    plt.figure(figsize=(8, 4))
    plt.bar(detectors, df_compare["Δ_Accuracy"], color='green', alpha=0.7)
    plt.axhline(0, color='black', linewidth=0.8)
    plt.ylabel("Δ Accuracy (V2 - V1)")
    plt.title("Performance Improvement per Detector")
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
    print(f"Finished analysis for dataset: {dataset2}")
    print("Mean V1 Accuracy:", df_compare["V1_Accuracy"].mean())
    print("Mean V2 Accuracy:", df_compare["V2_Accuracy"].mean())
    print("Mean V3 Accuracy:", df_compare["V3_Accuracy"].mean())
    print("Mean V4 Accuracy:", df_compare["V4_Accuracy"].mean())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# List of all datasets to process
datasets_to_process = ["bigger_mnist", "bigger_emnist", "coil100", "tu_berlin", "modelnet10"]
detectors_filtered = ["knn", "knn_mixed", "vim"]

# Store results for all datasets
all_results = {}

for dataset_vis in datasets_to_process:
    dataset2 = dataset_vis
    architecture = default_architecutre_mapping[dataset2]
    dataset_info2 = get_dataset_info(dataset2)

    # === LOAD RESULTS ===
    def load_metrics(result_dir, detectors):
        results = {}
        for det in detectors:
            eval_path = os.path.join(result_dir, det, "eval_results.json")
            if os.path.exists(eval_path):
                try:
                    with open(eval_path, "r") as f:
                        data = json.load(f)
                        results[det] = {
                            "accuracy_mean": data.get("accuracy_mean", np.nan),
                            "accuracy_std": data.get("accuracy_std", np.nan),
                        }
                except Exception as e:
                    print(f"Error loading {eval_path}: {e}")
            else:
                print(f"File not found: {eval_path}")
        return results

    metrics_v1 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v1_one_loader",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors_filtered,
    )
    metrics_v2 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v2_two_loaders",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors_filtered,
    )
    metrics_v3 = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_v3_three_loaders",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors_filtered,
    )
    metrics_halving = load_metrics(
        os.path.join(
            current_path,
            "experiment_files",
            "ood_studies_halving_full_loader",
            str(dataset2),
            str(architecture),
            getattr(dataset_info2, "transform_seq_name", "default"),
        ),
        detectors_filtered,
    )

    # === CREATE COMPARISON TABLE ===
    comparison_data = []
    for det in detectors_filtered:
        v1 = metrics_v1.get(det, {"accuracy_mean": np.nan, "accuracy_std": np.nan})
        v2 = metrics_v2.get(det, {"accuracy_mean": np.nan, "accuracy_std": np.nan})
        v3 = metrics_v3.get(det, {"accuracy_mean": np.nan, "accuracy_std": np.nan})
        v4 = metrics_halving.get(det, {"accuracy_mean": np.nan, "accuracy_std": np.nan})

        comparison_data.append({
            "Detector": det,
            "V1_Accuracy": v1["accuracy_mean"],
            "V1_Std": v1["accuracy_std"],
            "V2_Accuracy": v2["accuracy_mean"],
            "V2_Std": v2["accuracy_std"],
            "V3_Accuracy": v3["accuracy_mean"],
            "V3_Std": v3["accuracy_std"],
            "V4_Accuracy": v4["accuracy_mean"],
            "V4_Std": v4["accuracy_std"],
        })

    df_compare = pd.DataFrame(comparison_data)
    all_results[dataset_vis] = df_compare

    print(f"\n=== OOD Detector Comparison ({dataset_vis}) ===")
    print(df_compare.to_string(index=False))

# === GENERATE PLOTS FOR ALL DATASETS ===
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, dataset_vis in enumerate(datasets_to_process):
    df_compare = all_results[dataset_vis]
    architecture = default_architecutre_mapping[dataset_vis]

    x = np.arange(len(detectors_filtered))
    width = 0.2

    ax = axes[idx]
    ax.bar(x - 1.5*width, df_compare["V1_Accuracy"], width, label='V1', color='skyblue', alpha=0.7)
    ax.bar(x - 0.5*width, df_compare["V2_Accuracy"], width, label='V2', color='salmon', alpha=0.7)
    ax.bar(x + 0.5*width, df_compare["V3_Accuracy"], width, label='V3', color='lightgreen', alpha=0.7)
    ax.bar(x + 1.5*width, df_compare["V4_Accuracy"], width, label='Halving', color='orange', alpha=0.7)

    ax.set_ylabel('Accuracy')
    ax.set_title(f'{dataset_vis} ({architecture})')
    ax.set_xticks(x)
    ax.set_xticklabels(detectors_filtered, rotation=0)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_ylim(0, 1)

# Hide the extra subplot
axes[-1].axis('off')

plt.suptitle('OOD Detector Accuracy Comparison (All Datasets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# === PRINT SUMMARY STATISTICS ===
print("\n=== SUMMARY STATISTICS ===")
for dataset_vis in datasets_to_process:
    df_compare = all_results[dataset_vis]
    print(f"\n{dataset_vis.upper()}:")
    print(f"  Mean V1 Accuracy: {df_compare['V1_Accuracy'].mean():.4f}")
    print(f"  Mean V2 Accuracy: {df_compare['V2_Accuracy'].mean():.4f}")
    print(f"  Mean V3 Accuracy: {df_compare['V3_Accuracy'].mean():.4f}")
    print(f"  Mean V4 Accuracy: {df_compare['V4_Accuracy'].mean():.4f}")

In [ ]:
# === GENERATE PLOTS FOR ALL DATASETS WITH ERROR BARS ===
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, dataset_vis in enumerate(datasets_to_process):
    df_compare = all_results[dataset_vis]
    architecture = default_architecutre_mapping[dataset_vis]

    x = np.arange(len(detectors_filtered))
    width = 0.2

    ax = axes[idx]
    # Bars with error bars
    ax.bar(x - 1.5*width, df_compare["V1_Accuracy"], width, label='V1', color='skyblue', alpha=0.7,
           yerr=df_compare["V1_Std"], capsize=5, error_kw={'elinewidth':1, 'ecolor':'blue'})
    ax.bar(x - 0.5*width, df_compare["V2_Accuracy"], width, label='V2', color='salmon', alpha=0.7,
           yerr=df_compare["V2_Std"], capsize=5, error_kw={'elinewidth':1, 'ecolor':'red'})
    ax.bar(x + 0.5*width, df_compare["V3_Accuracy"], width, label='V3', color='lightgreen', alpha=0.7,
           yerr=df_compare["V3_Std"], capsize=5, error_kw={'elinewidth':1, 'ecolor':'green'})
    ax.bar(x + 1.5*width, df_compare["V4_Accuracy"], width, label='Halving', color='orange', alpha=0.7,
           yerr=df_compare["V4_Std"], capsize=5, error_kw={'elinewidth':1, 'ecolor':'darkorange'})

    ax.set_ylabel('Accuracy')
    ax.set_title(f'{dataset_vis} ({architecture})')
    ax.set_xticks(x)
    ax.set_xticklabels(detectors_filtered, rotation=0)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_ylim(0, 1)

# Hide the extra subplot if there is one
if len(datasets_to_process) < len(axes):
    for extra_ax in axes[len(datasets_to_process):]:
        extra_ax.axis('off')

plt.suptitle('OOD Detector Accuracy Comparison (All Datasets)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# === CALCULATE MEANS PER DATASET WITH DESCRIPTIVE NAMES ===
summary_data = []

method_names = ["1 Val Set", "2 Val Sets", "3 Val Sets", "Halving"]

for dataset_vis in datasets_to_process:
    df_compare = all_results[dataset_vis]
    summary_data.append({
        "Dataset": dataset_vis,
        "1 Val Set": df_compare["V1_Accuracy"].mean(),
        "2 Val Sets": df_compare["V2_Accuracy"].mean(),
        "3 Val Sets": df_compare["V3_Accuracy"].mean(),
        "Halving": df_compare["V4_Accuracy"].mean(),
    })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_data)

# === CALCULATE OVERALL MEAN ACROSS ALL DATASETS ===
overall_means = summary_df[method_names].mean()
overall_row = {"Dataset": "Overall"}
overall_row.update(overall_means.to_dict())
summary_df = pd.concat([summary_df, pd.DataFrame([overall_row])], ignore_index=True)

# === MARK THE BEST VALUE PER ROW WITH LaTeX BOLD ===
def highlight_best(row):
    row_copy = row.copy()
    # Skip the Dataset column
    values = row[method_names]
    max_val = values.max()
    for method in method_names:
        if values[method] == max_val:
            row_copy[method] = f"\\textbf{{{values[method]:.4f}}}"
        else:
            row_copy[method] = f"{values[method]:.4f}"
    # Keep the Dataset column unchanged
    row_copy["Dataset"] = row["Dataset"]
    return row_copy

highlighted_df = summary_df.apply(highlight_best, axis=1)

# Export to LaTeX
latex_table = highlighted_df.to_latex(index=False, escape=False,
                                      caption="Mean Accuracy per Dataset with Different Validation Strategies (Best in Bold)",
                                      label="tab:mean_accuracy")
print(latex_table)
